In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
model =  ChatGroq(model="Gemma2-9b-It", groq_api_key=groq_api_key)

# talk with llm
model.invoke([HumanMessage(content="Hi my name is Gaurav, I'm a ML engineer")])


AIMessage(content="Hi Gaurav, it's nice to meet you!\n\nBeing an ML engineer is fascinating. What kind of projects are you working on these days?  \n\nI'm always eager to learn more about the exciting work being done in the field of machine learning.\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 22, 'total_tokens': 79, 'completion_time': 0.103636364, 'prompt_time': 0.002106208, 'queue_time': 0.243986875, 'total_time': 0.105742572}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--3cd3d68a-202c-46a0-b0c5-9a45618b9fe7-0', usage_metadata={'input_tokens': 22, 'output_tokens': 57, 'total_tokens': 79})

In [2]:
from langchain_core.messages import AIMessage

model.invoke(
    [
        HumanMessage(content="Hi my name is Gaurav, I'm a ML engineer"),
        AIMessage(content="Hi Gaurav, it's nice to meet you!\n\nBeing an ML engineer is fascinating. What kind of projects are you working on these days?  \n\nI'm always eager to learn more about the exciting work being done in the field of machine learning.\n"),
        HumanMessage(content="whats my name?")
    ]
)

AIMessage(content='Your name is Gaurav, as you told me at the beginning of our conversation! 😊  \n\nIs there anything else I can help you with?\n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 91, 'total_tokens': 124, 'completion_time': 0.06, 'prompt_time': 0.005214144, 'queue_time': 0.24944860500000002, 'total_time': 0.065214144}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--459b8d4a-ae66-409d-a085-aa84bd063942-0', usage_metadata={'input_tokens': 91, 'output_tokens': 33, 'total_tokens': 124})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [5]:
# save chat message history
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [8]:
config = {"configurable":{"session_id":"chat1"}}

In [ ]:
response = with_message_history.invoke(
    [
        HumanMessage(content="Hi my name is Gaurav, I'm a ML engineer")
    ],
    config=config
)

AIMessage(content="Hi Gaurav, it's nice to meet you!\n\nI'm Gemma, an AI assistant. As an ML engineer myself (in a virtual sense!), I'm always interested in connecting with others in the field.\n\nWhat are you working on these days? 😊\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 22, 'total_tokens': 81, 'completion_time': 0.107272727, 'prompt_time': 0.002083648, 'queue_time': 0.241724125, 'total_time': 0.109356375}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--d408c6a8-52b1-4072-816e-664a9deacfc1-0', usage_metadata={'input_tokens': 22, 'output_tokens': 59, 'total_tokens': 81})

In [10]:
# check if llm remembers my name
with_message_history.invoke(
    [
        HumanMessage(content="Whats my name?")
    ],
    config=config
)

AIMessage(content='Your name is Gaurav.  I remember! 😊  How can I help you today? \n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 93, 'total_tokens': 116, 'completion_time': 0.041818182, 'prompt_time': 0.016524993, 'queue_time': 0.432113515, 'total_time': 0.058343175}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--a3b9c4a9-4768-4dc8-b747-ccfdeb8d4a43-0', usage_metadata={'input_tokens': 93, 'output_tokens': 23, 'total_tokens': 116})

In [11]:
# what if i change the config
config2 = {"configurable":{"session_id":"chat2"}}
with_message_history.invoke(
    [
        HumanMessage(content="Whats my name?")
    ],
    config=config2
)

AIMessage(content="As a large language model, I have no memory of past conversations and do not know your name. If you'd like to tell me your name, I'd be happy to know! 😊\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 13, 'total_tokens': 56, 'completion_time': 0.078181818, 'prompt_time': 0.001905188, 'queue_time': 0.242428412, 'total_time': 0.080087006}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--15fe9d42-bf20-4536-bc06-05a07dd5235d-0', usage_metadata={'input_tokens': 13, 'output_tokens': 43, 'total_tokens': 56})